# DiffusionNFT：原始 NFT 与轨迹 Hybrid-NFT

本笔记严格按以下顺序组织：先定义原始 NFT loss；再区分 Flow Matching 与 Score Matching；然后将两种模型的预测统一转换到 $x_{start}$ 并实现原始 NFT；最后说明轨迹 Hybrid Loss 替换原始 MSE 的位置，并给出 Flow/Score 两个 Hybrid-NFT 实现。

本文代码针对轨迹张量 $(B,T,D)$，其中前两维是 $(x,y)$。图像 latent $(B,C,H,W)$ 没有时间轴，不能直接使用 waypoint 积分项。

# 1. 原始 NFT Loss

设旧策略预测为 $h_{old}$，当前可训练策略为 $h_\theta$，奖励归一化为最优性概率 $r\in[0,1]$。NFT 不训练两个独立网络，而是构造隐式正、负策略：

$$
h_\theta^+=(1-\beta)h_{old}+\beta h_\theta,\qquad h_\theta^-=(1+\beta)h_{old}-\beta h_\theta.
$$

给定干净目标 $x_{start}$、带噪状态 $x_t$ 和模型预测参数化 $h$，原始 NFT 对两个隐式分身分别做回归。本文沿用 HDP 的数据表示：$x_{start}$ 是干净的速度/位移增量轨迹，绝对位置轨迹另记为 $\tau_0^x$：

$$
L_+=\operatorname{MSE}(\hat x_{start}^+,x_{start}),\qquad L_-=\operatorname{MSE}(\hat x_{start}^-,x_{start}).
$$

奖励加权策略损失为：

$$
L_{NFT}=\mathbb E\left[\frac{r}{\beta}L_+ + \frac{1-r}{\beta}L_-\right].
$$

高奖励样本主要优化正分身，低奖励样本主要优化负分身。$h_{ref}$ 的 KL/MSE 正则独立加入：

$$
L_{total}=L_{NFT}+\beta_{KL}\operatorname{MSE}(h_\theta,h_{ref}).
$$

代码中的 $h_{old}$ 和 $h_{ref}$ 必须 stop-gradient 或不参与当前计算图；只有 $h_\theta$ 接收 NFT 策略梯度。

# 2. Flow Matching 与 Score Matching 的区别

NFT 的隐式正负组合、奖励权重和 KL 正则在两种模型中完全相同；必须区分的是模型输出到干净数据 $x_1$ 的转换。Hybrid loss 也必须在这个转换之后计算。

## Flow Matching

Flow Matching 使用条件最优传输（Conditional Optimal Transport, COT）直线路径及其速度场 $u_t$：

$$
x_t=(1-t)x_0+t x_1,\qquad u_t=\frac{d x_t}{dt}=x_1-x_0.
$$

因此：

$$
\hat x_1=x_t+(1-t)\hat u_t.
$$

这里的 $t\in[0,1]$ 是从源分布 $x_0$ 到真实数据 $x_1$ 的路径插值系数；因此 Flow 模型预测的是速度向量场，训练目标中的干净规划轨迹是 $x_1$。

## Score Matching / VP Diffusion

VP 扩散满足：

$$
x_t=\alpha_t x_0+\sigma_t\epsilon.
$$

不同预测头都先还原为干净数据 $x_1$：

$$
\hat x_1^{\epsilon}=\frac{x_t-\sigma_t\hat\epsilon}{\alpha_t},\qquad \hat x_1^{score}=\frac{x_t+\sigma_t^2\hat s}{\alpha_t},
$$
$$
\hat x_1^{v}=\frac{x_t-\sigma_t(\sigma_t x_t+\alpha_t\hat v)}{\alpha_t}.
$$

因此不能把 Flow 的路径时间 $t$ 当成 VP 的噪声标准差 $\sigma_t$，也不能把 flow velocity 当成 diffusion-v。统一原则是：先按各自物理过程转换到 $x_1$，再计算 NFT 的回归损失或轨迹 Hybrid loss。

# 3. Hybrid Loss 替换原始 NFT 的哪一部分？

原始 NFT 的替换边界只有一处：将每个隐式分支在 velocity-space $x_{start}$ 上的 MSE 替换为轨迹 Hybrid loss；正/负分支构造和奖励加权不变。$x_{start}$ 不是绝对位置，而是干净速度/位移增量。

对一个分支，转换得到的 $x_{start}$ 已经是速度/位移增量：

$$
\hat\tau_0^v=\hat x_{start},\qquad\tau_0^v=x_{start}.
$$

Hybrid 分支损失为：

$$
L_{branch}^{hybrid}=\operatorname{MSE}(\hat\tau_0^v,\tau_0^v)+\omega\operatorname{MSE}(M\hat\tau_0^v\Delta t,\tau_0^x).
$$

其中 $M$ 是下三角全 1 积分矩阵。第一项保证局部时间连续性，第二项抵消速度积分造成的全局漂移。于是：

$$
L_{NFT}^{hybrid}=\mathbb E\left[\frac{r}{\beta}L_{pos}^{hybrid}+\frac{1-r}{\beta}L_{neg}^{hybrid}\right].
$$

若令速度误差为 $e$，则单个分支满足：

$$
L_{branch}^{hybrid}=e^T(I+\omega\Delta t^2M^TM)e=\|e\|_P^2.
$$

代码实现先把每个正/负预测转换到 velocity-space $x_{start}$，直接计算第一项，再对 $x_{start}[..., :2]$ 积分计算 waypoint 项。因此 Flow 与 Score 版本的 Hybrid 计算部分完全相同，只有前面的 $x_{start}$ 转换不同。

# 4. 四个实现的调用关系与理论性质

原始 Flow NFT 调用函数 $\texttt{nft\_loss\_original\_flow}$；原始 Score/VP NFT 调用 $\texttt{nft\_loss\_original\_score}$，并通过 $prediction\_type$ 选择 $x_0/\epsilon/score/v$。替换后分别调用 $\texttt{nft\_loss\_hybrid\_flow}$ 与 $\texttt{nft\_loss\_hybrid\_score}$。

完整 Hybrid 积分时，正、负分支只是把欧氏度量 $I$ 换成相同的正定度量：

$$
P=I+\omega\Delta t^2M^TM.
$$

由于 $P$ 与样本奖励、正负分支及待优化输出无关，对 NFT 条件风险求导时，公共可逆矩阵 $P$ 可以从一阶条件中消去。因此完整 Hybrid-NFT 与原始 NFT 的理论最优强化方向相同，但额外强调时间积分后的全局轨迹误差。启用 $detach\_window\_size$ 后，前向 loss 数值不变，反向改为局部窗口代理梯度，此时应将其理解为训练稳定化近似。

若训练数据已归一化，必须传入 $state\_mean=[0,0,0,0]$、$state\_std=[0.5,0.5,1,1]$。直接速度项仍在归一化扩散空间计算；积分前先反归一化为物理增量。

In [1]:
from typing import Dict, Optional, Tuple
import torch
from torch import Tensor
import json
from mmengine import fileio

def openjson(path):
    """读取 JSON 文件并返回解析后的字典"""
    value = fileio.get_text(path)
    dict_data = json.loads(value)
    return dict_data

class StateNormalizer:
    """状态归一化器：提供归一化和反归一化操作，用于将物理量转换到归一化空间（或反之）"""
    def __init__(self, mean, std):
        # mean, std 形状都是 (4,)，对应 (dx, dy, cos, sin) 四个维度
        self.mean = torch.as_tensor(mean)
        self.std = torch.as_tensor(std)

    @classmethod
    def from_json(cls, args):
        """从配置文件（normalization.json）中读取均值和标准差，构建实例"""
        data = openjson(args.normalization_file_path)
        mean = [data["ego"]["mean"]]      # 例如 [0,0,0,0]
        std = [data["ego"]["std"]]        # 例如 [0.5,0.5,1,1]
        return cls(mean, std)
    
    def __call__(self, data):
        """归一化：z = (x - mean) / std
        data 形状 (..., 4)，返回同形状张量"""
        return (data - self.mean.to(data.device)) / self.std.to(data.device)

    def inverse(self, data):
        """反归一化：x = z * std + mean
        data 形状 (..., 4)，返回同形状张量"""
        return data * self.std.to(data.device) + self.mean.to(data.device)

    def to_dict(self):
        """导出为字典（用于保存配置）"""
        return {
            "mean": self.mean.detach().cpu().numpy().tolist(),
            "std": self.std.detach().cpu().numpy().tolist()
        }

# -----------------------------------------------------------------------------
# 基础工具函数
# -----------------------------------------------------------------------------

def _batch_scalar_shape(x: Tensor) -> Tuple[int, ...]:
    """
    返回将批标量广播到与 x 兼容的形状，例如 (B,) -> (B,1,1)。
    用于将时间步 t、alpha、sigma 等标量从 (B,) 广播到 (B,T,D)。
    """
    return (x.shape[0],) + (1,) * (x.ndim - 1)


def _reward_probability(advantages: Tensor, adv_clip_max: float) -> Tensor:
    """
    将优势值（advantages）映射到 [0,1] 区间，作为最优性概率 r。
    输入 advantages: (B,)，输出 r: (B,)。
    步骤：截断 -> 线性映射 -> 再次截断。
    """
    a = advantages.clamp(-adv_clip_max, adv_clip_max)
    return ((a / adv_clip_max) / 2.0 + 0.5).clamp(0.0, 1.0)


# -----------------------------------------------------------------------------
# 加噪过程
# -----------------------------------------------------------------------------

def flow_add_noise(x1: Tensor, t: Tensor, noise: Optional[Tensor] = None) -> Tensor:
    """
    Flow Matching (COT) 加噪：x_t = (1-t)*noise + t*x1。
    输入 x1: (B,T,D)，t: (B,)，noise 可选 (B,T,D)。
    输出 x_t: (B,T,D)。
    形状变换：t 被 reshape 为 (B,1,1) 以便广播。
    """
    if noise is None:
        noise = torch.randn_like(x1)
    t = t.reshape(_batch_scalar_shape(x1))   # (B,) -> (B,1,1)
    return (1.0 - t) * noise + t * x1


def vp_add_noise(x1: Tensor, alpha: Tensor, sigma: Tensor, noise: Optional[Tensor] = None) -> Tensor:
    """
    VP-SDE 加噪：x_t = alpha*x1 + sigma*noise。
    输入 x1: (B,T,D)，alpha, sigma: (B,)，noise 可选。
    输出 x_t: (B,T,D)。
    形状变换：alpha, sigma 被 reshape 为 (B,1,1)。
    """
    if noise is None:
        noise = torch.randn_like(x1)
    alpha = alpha.reshape(_batch_scalar_shape(x1))   # (B,1,1)
    sigma = sigma.reshape(_batch_scalar_shape(x1))   # (B,1,1)
    return alpha * x1 + sigma * noise


# -----------------------------------------------------------------------------
# 模型预测到干净数据 x_start 的转换
# -----------------------------------------------------------------------------

def flow_prediction_to_x1(pred: Tensor, x_t: Tensor, t: Tensor) -> Tensor:
    """
    Flow 模型预测的速度场 v 转换为干净数据 x1。
    公式：x1 = x_t + (1-t)*v。
    输入 pred: (B,T,D) 速度场，x_t: (B,T,D)，t: (B,)。
    输出 x1: (B,T,D)。
    形状变换：t reshape 为 (B,1,1)。
    """
    t = t.reshape(_batch_scalar_shape(pred))   # (B,1,1)
    return x_t + (1.0 - t) * pred


def vp_prediction_to_x1(pred: Tensor, x_t: Tensor, alpha: Tensor, sigma: Tensor,
                        prediction_type: str = 'x0') -> Tensor:
    """
    VP-SDE 各种预测头转换为干净数据 x1。
    支持 prediction_type:
        'x0'    : 直接预测 x1
        'noise' : 预测噪声 eps
        'score' : 预测得分 s = -eps/sigma
        'v'     : 预测 v = alpha*eps - sigma*x1
    输入 pred, x_t: (B,T,D)，alpha, sigma: (B,)。
    输出 x1: (B,T,D)。
    形状变换：alpha, sigma reshape 为 (B,1,1)。
    """
    shape = _batch_scalar_shape(pred)          # (B,1,1)
    alpha = alpha.reshape(shape)
    sigma = sigma.reshape(shape)

    if prediction_type == 'x0':
        return pred
    elif prediction_type == 'noise':
        eps = pred
    elif prediction_type == 'score':
        eps = -sigma * pred
    elif prediction_type == 'v':
        # 利用 alpha^2 + sigma^2 = 1 解出 eps
        eps = sigma * x_t + alpha * pred
    else:
        raise ValueError(f'Unknown VP prediction_type: {prediction_type}')

    # x1 = (x_t - sigma * eps) / alpha
    return (x_t - sigma * eps) / (alpha + 1e-6)


ModuleNotFoundError: No module named 'mmengine'

In [ ]:
from typing import Optional, Tuple
import torch
from torch import Tensor


def detached_integral_recent_window(
    values: Tensor,
    window_size: int,
) -> Tensor:
    """
    对输入值沿时间维度进行积分（累积和），但反向传播时只允许梯度通过最近 window_size 个时间步。
    前向输出与普通 cumsum 完全一致，仅梯度被截断。

    参数:
        values: (B, T, D) 或任意形状 (..., T, D)，时间维度为 -2 位置
        window_size: 梯度回传的窗口大小（正整数）
    返回:
        integrated: 与 values 形状相同的积分结果，前向等于 values.cumsum(dim=-2)
    """
    if values.ndim < 2:
        raise ValueError("detached integral requires a time and feature dimension")
    if window_size < 1:
        raise ValueError(f"window_size must be positive, got {window_size}")

    horizon = values.shape[-2]               # 时间长度 T
    window = min(window_size, horizon)
    full_sum = values.cumsum(dim=-2)         # 完整累积和，形状同 values

    if window == horizon:
        return full_sum

    # 最近 window 步的和（保留梯度）
    recent = torch.cat(
        [
            full_sum[..., :window, :],                          # 前 window 步
            full_sum[..., window:, :] - full_sum[..., :-window, :],
        ],
        dim=-2,
    )  # 形状同 values

    # 窗口之前的历史（梯度截断）
    detached_history = torch.cat(
        [
            torch.zeros_like(full_sum[..., :window, :]),        # 前 window 步无历史
            full_sum[..., :-window, :].detach(),
        ],
        dim=-2,
    )  # 形状同 values

    # 前向：detached_history + recent = full_sum（数值一致）
    # 反向：梯度只通过 recent，因此只回传到最近 window 个输入
    return detached_history + recent


def reconstruction_loss_per_sample(
    error: Tensor,
    norm,
    *,
    loss_type: str = "mse",
    hybrid_weight: float = 0.1,
    hybrid_detach_window: Optional[int] = None,
) -> Tensor:
    """
    计算逐样本的重建损失，返回形状 (B,)。

    参数:
        error: (B, T, D) 归一化空间中的误差（预测 - 目标）
        norm: StateNormalizer 实例，用于 Hybrid 中反归一化积分部分（取 norm.std）
        loss_type: "mse" 或 "hybrid"
        hybrid_weight: Hybrid 中 waypoint 项的权重 ω
        hybrid_detach_window: 梯度截断窗口大小。若为 None 或 >= T，则使用完整梯度；
                              若为正整数，则前向数值不变，反向梯度只回传最近窗口。
    返回:
        per_sample_loss: (B,) 每个样本的标量损失，已除以 (T*D) 对齐 MSE
    """
    if error.ndim != 3:
        raise ValueError(f"error must have shape (B, T, D), got {error.shape}")
    if loss_type not in {"mse", "hybrid"}:
        raise ValueError(f"Unknown loss_type: {loss_type}")
    if hybrid_weight < 0:
        raise ValueError(f"hybrid_weight must be non-negative, got {hybrid_weight}")
    if hybrid_detach_window is not None and hybrid_detach_window < 1:
        raise ValueError("hybrid_detach_window must be positive or None")

    B, T, D = error.shape

    # ---------- MSE 情况：直接返回逐样本均方误差 ----------
    if loss_type == "mse":
        return error.square().mean(dim=(1, 2))   # (B,)

    # ---------- Hybrid 情况 ----------
    if D < 2:
        raise ValueError("Hybrid loss requires at least 2 features (dx, dy)")

    # 提取各维度标准差（用于反归一化 dx/dy）
    std = norm.std.to(error).reshape(-1)  # (D,)
    if std.numel() != D:
        raise ValueError(f"normalizer std has {std.numel()} features, expected {D}")
    std_xy = std[:2]  # 仅前两维参与积分

    # 反归一化得到物理位移增量误差（仅 dx,dy）
    physical_delta_error = error[:, :, :2] * std_xy  # (B, T, 2)

    # 积分（若 hybrid_detach_window 为 None 或 >= T，则完整积分，等价于标准 P 范数）
    # 当窗口小于 T 时，前向数值不变，但反向梯度被截断
    if hybrid_detach_window is None:
        # 完整积分，不截断
        integrated = physical_delta_error.cumsum(dim=1)  # (B, T, 2)
    else:
        integrated = detached_integral_recent_window(
            physical_delta_error,
            hybrid_detach_window,
        )  # (B, T, 2)

    # 计算损失：action_loss 为所有维度平方和，waypoint_loss 为积分后位置误差平方和
    action_loss = error.square().sum(dim=(1, 2))          # (B,)
    waypoint_loss = integrated.square().sum(dim=(1, 2))   # (B,)

    # 总损失，除以 (T * D) 使尺度与 MSE 一致
    loss = (action_loss + hybrid_weight * waypoint_loss) / (T * D)  # (B,)
    return loss

In [ ]:
# -----------------------------------------------------------------------------
# 统一的 DiffusionNFT 损失入口
# -----------------------------------------------------------------------------

def nft_loss(
    pred_theta: Tensor,
    pred_old: Tensor,
    pred_ref: Tensor,
    target_x_start: Tensor,
    x_t: Tensor,
    t_or_alpha: Tensor,
    sigma: Optional[Tensor] = None,
    advantages: Optional[Tensor] = None,
    beta: float = 0.5,
    adv_clip_max: float = 1.0,
    beta_kl_local: float = 0.1,
    beta_kl_global: float = 0.0,
    model_type: str = 'flow',
    prediction_type: str = 'x0',
    loss_type: str = 'original',
    omega: float = 0.1,
    detach_window_size: Optional[int] = None,
    norm=None,
    adaptive_scale: bool = True,
) -> Tuple[Tensor, Dict[str, Tensor]]:
    """
    DiffusionNFT 统一损失函数，支持 Flow 与 VP-Score，支持原始 MSE 或 Hybrid P 范数。

    参数说明:
        pred_theta: (B,T,D) 当前策略预测
        pred_old:   (B,T,D) 旧策略预测，需已 detach
        pred_ref:   (B,T,D) 参考模型预测，需已 detach
        target_x_start: (B,T,D) 干净数据目标（归一化空间）
        x_t:        (B,T,D) 加噪状态
        t_or_alpha: (B,) Flow 的时间 t 或 VP 的 alpha
        sigma:      (B,) VP 的 sigma，仅 VP 时必需
        advantages: (B,) 优势值；若 None 则 r=0.5
        beta:       隐式策略混合系数
        adv_clip_max: 优势截断范围
        beta_kl_local:  局部 KL 权重（约束与旧策略的偏差）
        beta_kl_global: 全局 KL 权重（约束与参考模型的偏差，通常为 0）
        model_type: 'flow' 或 'vp'
        prediction_type: VP 预测类型（'x0','noise','score','v'）
        loss_type: 'original'（MSE）或 'hybrid'（P 范数）
        omega:      Hybrid 中 waypoint 权重
        detach_window_size: Hybrid 截断梯度窗口
        norm:       StateNormalizer 实例，用于 Hybrid 反归一化
        adaptive_scale: 是否使用自适应加权（除以分支误差绝对值均值）

    返回:
        total_loss: 标量
        info: 各项损失字典（均已 detach）
    """
    if norm is None:
        raise ValueError("norm must be provided for target normalization and hybrid loss")
                         
    B = pred_theta.shape[0]
    device = pred_theta.device
    D = pred_theta.shape[-1]

    # 将未归一化的目标转换为归一化空间
    target_normalized = norm(target_x_start)   # (B,T,D) 归一化目标

    # 1. 奖励概率
    if advantages is None:
        r = torch.full((B,), 0.5, device=device)
    else:
        r = _reward_probability(advantages, adv_clip_max)  # (B,)

    # 2. 隐式正负策略（基于旧策略的对称外推）
    delta = pred_theta - pred_old.detach()                # (B,T,D)
    pos_pred = pred_old.detach() + beta * delta           # (B,T,D) 正策略
    neg_pred = pred_old.detach() - beta * delta           # (B,T,D) 负策略

    # 3. 转换到 x_start（干净数据）
    if model_type == 'flow':
        pos_x_start = flow_prediction_to_x1(pos_pred, x_t, t_or_alpha)
        neg_x_start = flow_prediction_to_x1(neg_pred, x_t, t_or_alpha)
    elif model_type == 'vp':
        if sigma is None:
            raise ValueError("sigma required for vp model")
        pos_x_start = vp_prediction_to_x1(pos_pred, x_t, t_or_alpha, sigma, prediction_type)
        neg_x_start = vp_prediction_to_x1(neg_pred, x_t, t_or_alpha, sigma, prediction_type)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    # 4. 计算分支误差（归一化空间）
    pos_err = pos_x_start - target_normalized   # (B,T,D)
    neg_err = neg_x_start - target_normalized   # (B,T,D)

    # 5. 调用统一的重建损失函数（返回 (B,) 逐样本损失）
    l_pos = reconstruction_loss_per_sample(
        pos_err,
        norm,
        loss_type=loss_type,
        hybrid_weight=omega,
        hybrid_detach_window=detach_window_size,
    )
    l_neg = reconstruction_loss_per_sample(
        neg_err,
        norm,
        loss_type=loss_type,
        hybrid_weight=omega,
        hybrid_detach_window=detach_window_size,
    )

    # 6. 可选：自适应加权（除以分支误差的绝对值均值）
    if adaptive_scale:
        with torch.no_grad():
            w_pos = pos_err.abs().mean(dim=(1, 2)).clamp(min=1e-5)  # (B,)
            w_neg = neg_err.abs().mean(dim=(1, 2)).clamp(min=1e-5)
        l_pos = l_pos / w_pos
        l_neg = l_neg / w_neg

    # 7. 策略损失（奖励加权）
    policy_loss = (r * l_pos + (1.0 - r) * l_neg) * adv_clip_max / beta  # (B,)

    # 8. 双 KL 正则化
    kl_local = ((pred_theta - pred_old.detach()) ** 2).mean()
    kl_global = ((pred_theta - pred_ref.detach()) ** 2).mean() if beta_kl_global > 0 else torch.tensor(0.0, device=device)

    total_loss = policy_loss + beta_kl_local * kl_local + beta_kl_global * kl_global

    # 9. 日志
    info = {
        'policy_loss': policy_loss.detach(),
        'kl_local': kl_local.detach(),
        'kl_global': kl_global.detach() if isinstance(kl_global, Tensor) else kl_global,
        'positive_loss': l_pos.mean().detach(),
        'negative_loss': l_neg.mean().detach(),
        'r_mean': r.mean().detach(),
        'total_loss': total_loss.detach(),
    }
    return total_loss, info

In [ ]:
# -----------------------------------------------------------------------------
# 便捷包装函数（匹配新的 nft_loss 签名）
# -----------------------------------------------------------------------------

def nft_loss_original_flow(
    pred_theta: Tensor,
    pred_old: Tensor,
    pred_ref: Tensor,
    target_x_start: Tensor,          # 未归一化的原始轨迹增量
    x_t: Tensor,
    t: Tensor,
    advantages: Tensor,
    beta: float,
    beta_kl: float,
    adv_clip_max: float,
    norm,                            # StateNormalizer 实例，用于归一化 target
) -> Tuple[Tensor, Dict[str, Tensor]]:
    """
    Flow Matching 原始 NFT 损失（使用自适应 MSE）。

    参数:
        pred_theta, pred_old, pred_ref: (B,T,D) 模型预测（归一化空间）
        target_x_start: (B,T,D) 未归一化目标
        x_t: (B,T,D) 加噪状态
        t: (B,) 时间步
        advantages: (B,) 优势值
        beta, beta_kl, adv_clip_max: 超参数
        norm: StateNormalizer 实例，用于归一化目标

    返回:
        total_loss, info
    """
    return nft_loss(
        pred_theta=pred_theta,
        pred_old=pred_old,
        pred_ref=pred_ref,
        target_x_start=target_x_start,
        x_t=x_t,
        t_or_alpha=t,
        sigma=None,
        advantages=advantages,
        beta=beta,
        adv_clip_max=adv_clip_max,
        beta_kl_local=beta_kl,
        beta_kl_global=0.0,
        model_type='flow',
        prediction_type='x0',
        loss_type='original',
        adaptive_scale=True,
        norm=norm,                   # 必须提供
    )


def nft_loss_original_score(
    pred_theta: Tensor,
    pred_old: Tensor,
    pred_ref: Tensor,
    target_x_start: Tensor,          # 未归一化目标
    x_t: Tensor,
    alpha: Tensor,
    sigma: Tensor,
    advantages: Tensor,
    beta: float,
    beta_kl: float,
    adv_clip_max: float,
    norm,                            # StateNormalizer 实例
    prediction_type: str = 'x0',
) -> Tuple[Tensor, Dict[str, Tensor]]:
    """
    VP/Score 原始 NFT 损失（使用自适应 MSE）。

    参数:
        pred_theta, pred_old, pred_ref: (B,T,D)
        target_x_start: (B,T,D) 未归一化目标
        x_t: (B,T,D) 加噪状态
        alpha, sigma: (B,) VP 噪声参数
        advantages: (B,)
        beta, beta_kl, adv_clip_max: 超参数
        norm: StateNormalizer 实例，用于归一化目标
        prediction_type: 模型预测类型

    返回:
        total_loss, info
    """
    return nft_loss(
        pred_theta=pred_theta,
        pred_old=pred_old,
        pred_ref=pred_ref,
        target_x_start=target_x_start,
        x_t=x_t,
        t_or_alpha=alpha,
        sigma=sigma,
        advantages=advantages,
        beta=beta,
        adv_clip_max=adv_clip_max,
        beta_kl_local=beta_kl,
        beta_kl_global=0.0,
        model_type='vp',
        prediction_type=prediction_type,
        loss_type='original',
        adaptive_scale=True,
        norm=norm,
    )


def nft_loss_hybrid_flow(
    pred_theta: Tensor,
    pred_old: Tensor,
    pred_ref: Tensor,
    target_x_start: Tensor,          # 未归一化目标
    x_t: Tensor,
    t: Tensor,
    advantages: Tensor,
    beta: float,
    beta_kl: float,
    adv_clip_max: float,
    norm,                            # StateNormalizer 实例
    omega: float = 0.1,
    detach_window_size: Optional[int] = None,
) -> Tuple[Tensor, Dict[str, Tensor]]:
    """
    Flow Matching 混合 NFT 损失（使用 Hybrid P 范数）。

    参数:
        pred_theta, pred_old, pred_ref: (B,T,D)
        target_x_start: (B,T,D) 未归一化目标
        x_t: (B,T,D) 加噪状态
        t: (B,) 时间步
        advantages: (B,)
        beta, beta_kl, adv_clip_max: 超参数
        norm: StateNormalizer 实例，用于归一化目标及 Hybrid 反归一化
        omega: Hybrid 中 waypoint 权重
        detach_window_size: 截断梯度窗口，None 为不截断

    返回:
        total_loss, info
    """
    return nft_loss(
        pred_theta=pred_theta,
        pred_old=pred_old,
        pred_ref=pred_ref,
        target_x_start=target_x_start,
        x_t=x_t,
        t_or_alpha=t,
        sigma=None,
        advantages=advantages,
        beta=beta,
        adv_clip_max=adv_clip_max,
        beta_kl_local=beta_kl,
        beta_kl_global=0.0,
        model_type='flow',
        prediction_type='x0',
        loss_type='hybrid',
        omega=omega,
        detach_window_size=detach_window_size,
        norm=norm,
        adaptive_scale=True,
    )


def nft_loss_hybrid_score(
    pred_theta: Tensor,
    pred_old: Tensor,
    pred_ref: Tensor,
    target_x_start: Tensor,          # 未归一化目标
    x_t: Tensor,
    alpha: Tensor,
    sigma: Tensor,
    advantages: Tensor,
    beta: float,
    beta_kl: float,
    adv_clip_max: float,
    norm,                            # StateNormalizer 实例
    prediction_type: str = 'x0',
    omega: float = 0.1,
    detach_window_size: Optional[int] = None,
) -> Tuple[Tensor, Dict[str, Tensor]]:
    """
    VP/Score 混合 NFT 损失（使用 Hybrid P 范数）。

    参数:
        pred_theta, pred_old, pred_ref: (B,T,D)
        target_x_start: (B,T,D) 未归一化目标
        x_t: (B,T,D) 加噪状态
        alpha, sigma: (B,) VP 噪声参数
        advantages: (B,)
        beta, beta_kl, adv_clip_max: 超参数
        norm: StateNormalizer 实例
        prediction_type: 模型预测类型
        omega: Hybrid 权重
        detach_window_size: 截断窗口

    返回:
        total_loss, info
    """
    return nft_loss(
        pred_theta=pred_theta,
        pred_old=pred_old,
        pred_ref=pred_ref,
        target_x_start=target_x_start,
        x_t=x_t,
        t_or_alpha=alpha,
        sigma=sigma,
        advantages=advantages,
        beta=beta,
        adv_clip_max=adv_clip_max,
        beta_kl_local=beta_kl,
        beta_kl_global=0.0,
        model_type='vp',
        prediction_type=prediction_type,
        loss_type='hybrid',
        omega=omega,
        detach_window_size=detach_window_size,
        norm=norm,
        adaptive_scale=True,
    )

In [ ]:
# -----------------------------------------------------------------------------
# 测试用例（匹配最新 nft_loss 签名）
# -----------------------------------------------------------------------------

# 设置随机种子
torch.manual_seed(42)

# 公共参数
B, T, D = 2, 5, 4      # batch size, time steps, feature dims (dx,dy,cos,sin)
device = torch.device('cpu')

# 生成模拟数据
pred_theta = torch.randn(B, T, D, device=device, requires_grad=True)  # 归一化空间中的预测
pred_old = torch.randn_like(pred_theta).detach()
pred_ref = torch.randn_like(pred_theta).detach()

# 构造未归一化的目标增量（模拟从位置差分得到的物理增量）
# 注意：这是原始物理尺度的数据，norm 会将其归一化
target_x_start_raw = torch.randn(B, T, D, device=device) * 2.0  # 假设物理增量尺度约为 2

# 归一化参数（与训练时一致）
norm = StateNormalizer(mean=[0.0, 0.0, 0.0, 0.0],
                       std=[0.5, 0.5, 1.0, 1.0])

advantages = torch.tensor([0.8, 0.2], device=device)  # 不同奖励

# ---------- Flow 模型测试 ----------
t = torch.rand(B, device=device)  # 时间步 (0,1)
# 加噪：x_t = (1-t)*noise + t*x1，其中 x1 是归一化后的目标
# 注意：加噪通常在归一化空间进行，所以先用 norm 归一化目标再生成 x_t
target_normalized = norm(target_x_start_raw)
x_t_flow = flow_add_noise(target_normalized, t)  # 加噪

# 原始 Flow NFT loss
loss, info = nft_loss_original_flow(
    pred_theta, pred_old, pred_ref,
    target_x_start_raw,          # 传入未归一化目标
    x_t_flow, t, advantages,
    beta=0.5, beta_kl=0.1, adv_clip_max=1.0,
    norm=norm                    # 必须提供 norm
)
print("Flow Original NFT Loss:", loss.item())
print("Info:", {k: round(v.item(), 4) for k, v in info.items()})

# Hybrid Flow NFT loss（使用截断梯度）
loss_h, info_h = nft_loss_hybrid_flow(
    pred_theta, pred_old, pred_ref,
    target_x_start_raw,          # 未归一化目标
    x_t_flow, t, advantages,
    beta=0.5, beta_kl=0.1, adv_clip_max=1.0,
    norm=norm,                   # 归一化对象
    omega=0.1,                   # Hybrid 权重
    detach_window_size=3         # 截断窗口
)
print("\nFlow Hybrid NFT Loss:", loss_h.item())
print("Info:", {k: round(v.item(), 4) for k, v in info_h.items()})

# ---------- VP-Score 模型测试 ----------
alpha = torch.rand(B, device=device) * 0.5 + 0.5  # 模拟 alpha_t
sigma = torch.sqrt(1 - alpha**2)                  # VP 关系
x_t_vp = vp_add_noise(target_normalized, alpha, sigma)  # 加噪

# 原始 VP NFT loss（预测噪声）
loss_vp, info_vp = nft_loss_original_score(
    pred_theta, pred_old, pred_ref,
    target_x_start_raw,
    x_t_vp, alpha, sigma, advantages,
    beta=0.5, beta_kl=0.1, adv_clip_max=1.0,
    norm=norm,
    prediction_type='noise'
)
print("\nVP Original NFT Loss (noise):", loss_vp.item())
print("Info:", {k: round(v.item(), 4) for k, v in info_vp.items()})

# Hybrid VP NFT loss（预测 x0，截断梯度）
loss_vph, info_vph = nft_loss_hybrid_score(
    pred_theta, pred_old, pred_ref,
    target_x_start_raw,
    x_t_vp, alpha, sigma, advantages,
    beta=0.5, beta_kl=0.1, adv_clip_max=1.0,
    norm=norm,
    prediction_type='x0',
    omega=0.1,
    detach_window_size=2
)
print("\nVP Hybrid NFT Loss (x0):", loss_vph.item())
print("Info:", {k: round(v.item(), 4) for k, v in info_vph.items()})

# 检查梯度流
print("\nGradient flow test:")
loss_h.backward()
print("pred_theta.grad norm:", pred_theta.grad.norm().item())